# Conatus Phronesis — merge + quantização GGUF pra rodar no Ollama

Funde o adapter LoRA com o base `Qwen/Qwen3-14B`, converte pra GGUF e quantiza.
Runtime: GPU (T4/L4) acelera o merge; a conversão/quantização em si roda em CPU.

**Este notebook é para o DEPLOY, não para avaliar o modelo.** Para medir se o 14B
melhorou o raciocínio, use as células de avaliação do `train_colab.ipynb`, que rodam o
modelo direto em `transformers` sobre o adapter — sem passar por GGUF. Assim o resultado
mede o modelo, e não o modelo mais o efeito da quantização. Vir para cá antes disso
mistura duas variáveis e você não saberá a qual atribuir uma eventual piora.

**Escolha da quantização.** Depende de onde o modelo vai rodar:

| Quant | Tamanho (14B) | Onde cabe |
|---|---|---|
| Q8_0 | ~15,7 GB | L4/A100 |
| Q6_K | ~12,1 GB | L4/A100 |
| Q5_K_M | ~10,5 GB | L4/A100 |
| **Q4_K_M** | **~9,0 GB** | **L4/A100 — padrão** |
| Q3_K_M | ~7,3 GB | GPU de 8 GB (perde qualidade) |
| IQ3_M | ~6,9 GB | GPU de 8 GB |

O default é `Q4_K_M`: é o ponto de equilíbrio usual entre tamanho e qualidade, e cabe
folgado nos 24 GB do L4. As linhas Q3/IQ3 só interessam se o destino voltar a ser uma
GPU de 8 GB — o próprio llama.cpp classifica a família Q3 como "low quality", então
não use por hábito.

**Pré-requisito**: secret `HF_TOKEN` no Colab (ícone de chave) com acesso ao repo
privado do adapter.


In [ ]:
# 1) Login HF
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

# Alvo atual do branch. Para refazer o A/B com o modelo anterior, troque para
# "Qwen/Qwen3-8B" e use o secret PHRONESIS_8B_ADAPTER_REPO.
BASE_MODEL = "Qwen/Qwen3-14B"
ADAPTER_REPO = userdata.get("PHRONESIS_14B_ADAPTER_REPO")
if not ADAPTER_REPO:
    raise ValueError("Configure o secret PHRONESIS_14B_ADAPTER_REPO com o repo do adapter 14B")
MERGED_DIR = "/content/merged"
GGUF_F16 = "/content/model-f16.gguf"

# Q4_K_M (~9GB no 14B): padrao de equilibrio tamanho/qualidade, cabe folgado no L4.
# Suba pra Q5_K_M/Q6_K se quiser mais fidelidade; desca pra Q3_K_M/IQ3_M so se o destino
# for uma GPU de 8GB. Ver a tabela na celula acima.
QUANT_TYPE = "Q4_K_M"
GGUF_QUANT = f"/content/model-{QUANT_TYPE}.gguf"


In [ ]:
# 2) Deps pro merge
%pip install -q -U transformers accelerate peft safetensors


In [ ]:
# 3) Merge: baixa base + adapter, funde os pesos, salva em formato HF (safetensors)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Carregando base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cpu")

print("Carregando adapter (repo privado)...")
model = PeftModel.from_pretrained(base, ADAPTER_REPO)

print("Fundindo LoRA nos pesos do base...")
model = model.merge_and_unload()

print(f"Salvando modelo fundido em {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merge concluido.")


In [ ]:
# 4) llama.cpp: clona e instala os requirements do script de conversao
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt


In [ ]:
# 5) Converte o modelo fundido (HF/safetensors) pra GGUF f16
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16


In [ ]:
# 6) Compila o llama.cpp (cmake) pra ter o binario de quantizacao
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4


In [ ]:
# 7) Quantiza o f16 pro tipo escolhido
!/content/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_QUANT} {QUANT_TYPE}

import os
size_gb = os.path.getsize(GGUF_QUANT) / (1024**3)
print(f"\nGGUF quantizado: {GGUF_QUANT} ({size_gb:.2f} GB)")
print("Confira com `ollama ps` depois de carregar: se size_vram == size, coube 100% na GPU.")


In [ ]:
# 8) Baixa o arquivo final pro seu computador
from google.colab import files
files.download(GGUF_QUANT)


## 9) (opcional) Persistir o GGUF no HF Hub em vez de baixar

Util se a conexao cair no meio do download, ou pra reusar depois sem regerar.


In [ ]:
# Opcional: sobe o GGUF pro mesmo repo privado (evita perder se o download falhar)
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj=GGUF_QUANT,
    path_in_repo=f"gguf/model-{QUANT_TYPE}.gguf",
    repo_id=ADAPTER_REPO,
    repo_type="model",
)
print("GGUF tambem disponivel em:", f"https://huggingface.co/{ADAPTER_REPO}/blob/main/gguf/model-{QUANT_TYPE}.gguf")


## 10) Rodar no Ollama

Com o arquivo `.gguf` em mãos, crie um `Modelfile` ao lado dele:

```
FROM ./model-Q4_K_M.gguf

PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
```

Depois:

```bash
ollama create phronesis-14b -f Modelfile
ollama run phronesis-14b
```

Confira que coube inteiro na GPU com `ollama ps`: se `size_vram` bater com `size`, está
100% na GPU. Se houver divisão CPU/GPU, o modelo transbordou e a latência vai sofrer —
quantize mais agressivamente ou sirva num host com mais VRAM.

**Sobre a decodificação**: `temperature 0.6` + `top_p 0.95` + `top_k 20` são os
parâmetros recomendados do Qwen3 (constam do `generation_config.json` oficial) e são os
mesmos que `src/eval_harness.py` usa, então a demo bate com o eval. O `Modelfile` antigo
trazia `temperature 0` (greedy) com uma nota afirmando que greedy era melhor — essa
conclusão é **anterior** ao experimento registrado em
`data/eval/thinking_8b_eval_notes.md`, que mediu greedy puro entrando em loop de
repetição em 4 de 16 itens. O loop some com sampling. Não volte pra `temperature 0` sem
reler aquelas notas. O `phronesis-4b` que está em produção hoje ainda usa `temperature 0`.

O GGUF já carrega o chat template do Qwen3 embutido (inclusive o formato `<tool_call>` e
o canal `<think>`), então o Ollama renderiza tools automaticamente via `/api/chat` com o
parâmetro `tools`.
